### Import

In [1]:
import re
import pickle 
import numpy as np
import pandas as pd
import datetime as dt

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
eicu = "PATH TO DATA/eICU/"
output = "../Data/csvExtract/"

### Read cohort hospital admission ids

In [3]:
cohort_subject_id = []

with open("../Data/Cohort/cohort1_subject_id.txt", "r") as f:
    for subject_id in f:
        cohort_subject_id.append(str(subject_id.strip()))
        
len(cohort_subject_id)

12677

In [4]:
cohort_hadm_id = []

with open("../Data/Cohort/cohort1_hadm_id.txt", "r") as f:
    for hadm_id in f:
        cohort_hadm_id.append(int(hadm_id.strip()))
        
len(cohort_hadm_id)

14420

In [5]:
cohort_icustay_id = []

with open("../Data/Cohort/cohort1_stay_id.txt", "r") as f:
    for stay_id in f:
        cohort_icustay_id.append(int(stay_id.strip()))
        
len(cohort_icustay_id)

16149

### patient

In [6]:
patients = pd.read_csv(eicu + "patient.csv")

patients = patients[['uniquepid', 'patienthealthsystemstayid', 'patientunitstayid', 'gender', 'age', 'ethnicity',
                     'admissionheight', 'admissionweight', 'apacheadmissiondx', 'unitdischargeoffset',
                     'hospitaldischargeoffset', 'unitdischargestatus', 'hospitaldischargestatus', 'hospitalid']]

In [7]:
patients.loc[patients.age == '> 89', 'age'] = 90
patients.loc[(patients.admissionheight >= 250) | (patients.admissionheight <= 75), 'admissionheight'] = np.nan
patients.loc[(patients.admissionweight >= 250) | (patients.admissionweight <= 15), 'admissionweight'] = np.nan

In [8]:
patients.loc[(patients.gender != 'Male') & (patients.gender != 'Female'), 'gender'] = 'Unknown'
g_map = {'Unknown': 0, 'Male': 1, 'Female': 2}

def transform_gender(gender_series):
    global g_map
    return {'gender': gender_series.fillna('').apply(lambda s: g_map[s] if s in g_map else g_map[''])}

patients.update(transform_gender(patients.gender))

In [9]:
def transform_race_into_id(df):
    
    df.ethnicity.fillna('nodx', inplace=True)
    dx_type = df.ethnicity.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['ethnicity'] = df['ethnicity'].map(dictionary)
    
    return df, dictionary

admissions, ethnicity_dictionary = transform_race_into_id(patients)

In [10]:
patients.loc[patients.unitdischargestatus.isnull(), 'unitdischargestatus'] = patients.hospitaldischargestatus

mortality_map = {'': -1, 'Alive': 0, 'Expired': 1}

def transform_unitdischargestatus(unitdischargestatus_series):
    global mortality_map
    return {'unitdischargestatus': unitdischargestatus_series.fillna('').apply(lambda s: mortality_map[s] if s in mortality_map else mortality_map[''])}

patients.update(transform_unitdischargestatus(patients.unitdischargestatus))

def hospitaldischargestatus(hospitaldischargestatus_series):
    global mortality_map
    return {'hospitaldischargestatus': hospitaldischargestatus_series.fillna('').apply(lambda s: mortality_map[s] if s in mortality_map else mortality_map[''])}

patients.update(hospitaldischargestatus(patients.hospitaldischargestatus))

In [11]:
patients = patients[patients.patientunitstayid.isin(cohort_icustay_id)]

### hospital

In [12]:
hospital = pd.read_csv(eicu + "hospital.csv")
patients = pd.merge(patients, hospital, on=['hospitalid'], how='left')
patients.drop(columns=['hospitalid', 'region'], inplace=True)

In [13]:
ACS = ['Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)', 
       'Infarction, acute myocardial (MI)', 'MI admitted > 24 hrs after onset of ischemia']

ChestPainUnknown = ['Chest pain, atypical (noncardiac chest pain)', 'Chest pain, epigastric', 
                    'Chest pain, musculoskeletal', 'Chest pain, respiratory', 'Chest pain, unknown origin']

CHF = ['Cardiomyopathy', 'CHF, congestive heart failure', 'Shock, cardiogenic']

CVOther = ['Angina, stable (asymp or stable pattern of symptoms w/meds)', 'Anomaly, cardiac congenital', 
           'Arteriovenous malformation, surgery for', 'Atrial Septal Defect (ASD) Repair', 
           'Cardiovascular medical, other', 'Cardiovascular surgery, other', 'Congenital Defect Repair (Other)',
           'Contusion, myocardial (include r/o)', 'Efffusion, pericardial', 'Endocarditis', 
           'Hypertension-pulmonary, primary/idiopathic', 'Monitoring, hemodynamic (pre-operative evaluation)',
           'Pericardial effusion/tamponade', 'Pericardiectomy (total/subtotal)', 'Pericarditis', 
           'Tamponade, pericardial', 'Thrombus, arterial', 'Vascular medical, other', 'Vascular surgery, other',
           'Ablation or mapping of cardiac conduction pathway', 
           'Defibrillator, automatic implantable cardiac; insertion of']

CardiacArrest = ['Cardiac arrest (with or without respiratory arrest; for respiratory arrest see Respiratory System)', 
                 'Rhythm disturbance (atrial, supraventricular)', 'Rhythm disturbance (conduction defect)', 
                 'Rhythm disturbance (ventricular)']

CABG = ['CABG alone, coronary artery bypass grafting', 'CABG alone, redo', 'CABG redo with other operation', 
        'CABG redo with valve repair/replacement', 'CABG with aortic valve replacement', 
        'CABG with double valve repair/replacement', 'CABG with mitral valve repair', 
        'CABG with mitral valve replacement', 'CABG with other operation', 
        'CABG with pulmonic or tricuspid valve repair or replacement ONLY.',
        'CABG with single valve repair/replacement', 'CABG, minimally invasive; mid-CABG']

ValveDz = ['Aortic and Mitral valve replacement', 'Aortic valve replacement (isolated)', 'Mitral valve repair', 
           'Mitral valve replacement', 'Papillary muscle rupture', 'Pulmonary valve surgery', 
           'Tricuspid valve surgery', 'Valve, double; repair/replacement', 'Valve, redo, single', 
           'Valve, single; repair/replacement', 'Valve, triple; repair/replacement']

PNA = ['Pneumonia, aspiration', 'Pneumonia, bacterial', 'Pneumonia, fungal', 'Pneumonia, other', 
       'Pneumonia, parasitic (i.e., Pneumocystic pneumonia)', 'Pneumonia, viral']

RespMedOther = ['Apnea, sleep', 'Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)', 
                'ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema', 
                'Arrest, respiratory (without cardiac arrest)']

RespMedOther = ['Atelectasis', 'Biopsy, open lung', 'Effusions, pleural', 'Embolus, pulmonary', 
                'Guillain-Barre syndrome', 'Hemorrhage/hemoptysis, pulmonary', 'Hemothorax', 
                'Obstruction-airway (i.e., acute epiglottitis, post-extubation edema, foreign body, etc)', 
                'Pneumothorax', 'Respiratory - medical, other',
                'Restrictive lung disease (i.e., Sarcoidosis, pulmonary fibrosis)', 'Tracheostomy', 
                'Weaning from mechanical ventilation (transfer from other unit or hospital only)']

Asthma_Emphys = ['Asthma', 'Emphysema/bronchitis']

GIBleed = ['Bleeding, GI from esophageal varices/portal hypertension', 'Bleeding, GI-location unknown', 
           'Bleeding, lower GI', 'Bleeding, upper GI', 'Bleeding-lower GI, surgery for',
           'Bleeding-other GI, surgery for', 'Bleeding-upper GI, surgery for', 
           'Bleeding-variceal, surgery for (excluding vascular shunting-see surgery for portosystemic shunt)',
           'GI perforation/rupture', 'GI perforation/rupture, surgery for', 'Hemorrhage, intra/retroperitoneal', 
           'Ulcer disease, peptic']

GIObstruction = ['GI obstruction', 'GI obstruction, surgery for (including lysis of adhesions)']

CVA = ['CVA, cerebrovascular accident/stroke', 'Hemorrhage/hematoma, intracranial', 
       'Hemorrhage/hematoma-intracranial, surgery for', 
       'Hypertension, uncontrolled (for cerebrovascular accident-see Neurological System)', 
       'Subarachnoid hemorrhage/arteriovenous malformation', 'Subarachnoid hemorrhage/intracranial aneurysm',
       'Subarachnoid hemorrhage/intracranial aneurysm, surgery for']

Neuro = ['Abscess, neurologic', 'Biopsy, brain', 'Hydrocephalus, obstructive', 'Neoplasm, neurologic', 
         'Neoplasm-cranial, surgery for (excluding transphenoidal)', 
         'Neoplasm-spinal cord, surgery or other related procedures', 'Neurologic medical, other', 
         'Neuromuscular medical, other', 'Palsy, cranial nerve', 'Seizures (primary-no structural brain disease)', 
         'Seizures-intractable, surgery for']

Coma = ['Coma/change in level of consciousness (for hepatic see GI, for diabetic see Endocrine, if related to cardiac arrest, see CV)', 
        'Nontraumatic coma due to anoxia/ischemia']

Overdose = ['Overdose, alcohols (bethanol, methanol, ethylene glycol)', 
            'Overdose, analgesic (aspirin, acetaminophen)', 'Overdose, antidepressants (cyclic, lithium)', 
            'Overdose, other toxin, poison or drug', 'Overdose, sedatives, hypnotics, antipsychotics, benzodiazepines', 
            'Overdose, self-inflicted', 'Overdose, street drugs (opiates, cocaine, amphetamine)', 
            'Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)']

Sepsis = ['Sepsis, cutaneous/soft tissue', 'Sepsis, GI', 'Sepsis, gynecologic', 'Sepsis, other', 
          'Sepsis, pulmonary', 'Sepsis, renal/UTI (including bladder)', 'Sepsis, unknown']

ARF = ['Renal failure, acute', 'Renal obstruction']

DKA = ['Diabetic hyperglycemic hyperosmolar nonketotic coma (HHNC)', 'Diabetic ketoacidosis']

Trauma_Chest = ['Abdomen only trauma', 'Abdomen/extremity trauma', 'Abdomen/face trauma', 'Abdomen/multiple trauma', 
                'Abdomen/pelvis trauma', 'Abdomen/spinal trauma', 'Chest thorax only trauma', 'Chest/abdomen trauma', 
                'Chest/extremity trauma', 'Chest/face trauma', 'Chest/multiple trauma', 'Chest/pelvis trauma', 
                'Chest/spinal trauma', 'Chest/thorax only trauma', 'Extremity only trauma']

Trauma_Face = ['Extremity only trauma, surgery for', 'Extremity/face trauma', 'Extremity/face trauma, surgery for', 
               'Extremity/multiple trauma', 'Extremity/multiple trauma, surgery for', 'Face only trauma', 
               'Face only trauma, surgery for', 'Face/multiple trauma', 'Face/multiple trauma, surgery for', 
               'Facial surgery (if related to trauma, see Trauma)', 'Head only trauma', 'Head/abdomen trauma', 
               'Head/chest trauma']

Trauma_Head = ['Head/extremity trauma', 'Head/face trauma', 'Head/multiple trauma', 'Head/pelvis trauma', 
               'Head/spinal trauma', 'Pelvis/extremity trauma', 'Pelvis/face trauma', 'Pelvis/hip trauma', 
               'Pelvis/multiple trauma', 'Pelvis/spinal trauma', 'Spinal cord only trauma', 'Spinal/extremity trauma', 
               'Spinal/face trauma', 'Spinal/multiple trauma', 'Trauma medical, other', 'Trauma surgery, other']

In [14]:
patients['apacheadmissioncategory'] = 'Others'

patients.loc[patients.apacheadmissiondx.isin(ACS),               'apacheadmissioncategory'] = 'ACS'
patients.loc[patients.apacheadmissiondx.isin(ChestPainUnknown),  'apacheadmissioncategory'] = 'ChestPainUnknown'
patients.loc[patients.apacheadmissiondx.isin(CHF),               'apacheadmissioncategory'] = 'CHF'
patients.loc[patients.apacheadmissiondx.isin(CVOther),           'apacheadmissioncategory'] = 'CVOther'
patients.loc[patients.apacheadmissiondx.isin(CardiacArrest),     'apacheadmissioncategory'] = 'CardiacArrest'
patients.loc[patients.apacheadmissiondx.isin(CABG),              'apacheadmissioncategory'] = 'CABG'
patients.loc[patients.apacheadmissiondx.isin(ValveDz),           'apacheadmissioncategory'] = 'ValveDz'
patients.loc[patients.apacheadmissiondx.isin(PNA),               'apacheadmissioncategory'] = 'PNA'
patients.loc[patients.apacheadmissiondx.isin(RespMedOther),      'apacheadmissioncategory'] = 'RespMedOther'
patients.loc[patients.apacheadmissiondx.isin(RespMedOther),      'apacheadmissioncategory'] = 'RespMedOther'
patients.loc[patients.apacheadmissiondx.isin(Asthma_Emphys),     'apacheadmissioncategory'] = 'Asthma_Emphys'
patients.loc[patients.apacheadmissiondx.isin(GIBleed),           'apacheadmissioncategory'] = 'GIBleed'
patients.loc[patients.apacheadmissiondx.isin(GIObstruction),     'apacheadmissioncategory'] = 'GIObstruction'
patients.loc[patients.apacheadmissiondx.isin(CVA),               'apacheadmissioncategory'] = 'CVA'
patients.loc[patients.apacheadmissiondx.isin(Neuro),             'apacheadmissioncategory'] = 'Neuro'
patients.loc[patients.apacheadmissiondx.isin(Coma),              'apacheadmissioncategory'] = 'Coma'
patients.loc[patients.apacheadmissiondx.isin(Overdose),          'apacheadmissioncategory'] = 'Overdose'
patients.loc[patients.apacheadmissiondx.isin(Sepsis),            'apacheadmissioncategory'] = 'Sepsis'
patients.loc[patients.apacheadmissiondx.isin(ARF),               'apacheadmissioncategory'] = 'ARF'
patients.loc[patients.apacheadmissiondx.isin(DKA),               'apacheadmissioncategory'] = 'DKA'
patients.loc[patients.apacheadmissiondx.isin(Trauma_Chest),      'apacheadmissioncategory'] = 'Trauma_Chest'
patients.loc[patients.apacheadmissiondx.isin(Trauma_Face),       'apacheadmissioncategory'] = 'Trauma_Face'
patients.loc[patients.apacheadmissiondx.isin(Trauma_Head),       'apacheadmissioncategory'] = 'Trauma_Head'

In [15]:
patients.head(3)

In [16]:
print(patients.uniquepid.nunique())
print(patients.patienthealthsystemstayid.nunique())
print(patients.patientunitstayid.nunique())
print(patients[patients.unitdischargestatus == 1].shape[0])
print(patients[patients.hospitaldischargestatus == 1].shape[0])

12677
14420
16149
1188
1856


### Apache

In [17]:
apacheApsVar = pd.read_csv(eicu + "apacheApsVar.csv")
apachePredVar = pd.read_csv(eicu + "apachePredVar.csv")
apachePatientResult = pd.read_csv(eicu + "apachePatientResult.csv")

In [18]:
apacheApsVar.drop(columns=['apacheapsvarid'], inplace=True)
apachePredVar = apachePredVar[['patientunitstayid', 'admitdiagnosis', 'age', 'thrombolytics', 'aids', 
                               'hepaticfailure', 'lymphoma', 'metastaticcancer', 'leukemia',
                               'immunosuppression', 'cirrhosis', 'electivesurgery', 'diabetes']] 
apachePatientResult.drop(columns=['apachepatientresultsid', 'physicianspeciality', 'physicianinterventioncategory',
                                  'preopmi', 'preopcardiaccath', 'ptcawithin24h', 'unabridgedunitlos',
                                  'unabridgedhosplos', 'actualventdays'], inplace=True)

In [19]:
apache_df = pd.merge(apachePredVar, apacheApsVar, on=['patientunitstayid'], how='left')
apache_df = pd.merge(apachePatientResult, apache_df, on=['patientunitstayid'], how='left')
apache_df = apache_df[apache_df.patientunitstayid.isin(cohort_icustay_id)]
apache_df = apache_df.reset_index(drop=True)

In [20]:
apache_df.head(3)

In [21]:
print(apache_df.patientunitstayid.nunique())
print(apache_df.shape)

14405
(28810, 50)


### pastHistory 

In [22]:
pastHistory = pd.read_csv(eicu + "pastHistory.csv")

In [23]:
pastHistory = pastHistory[['patientunitstayid', 'pasthistoryoffset', 'pasthistoryvalue']]
pastHistory.loc[pastHistory.pasthistoryoffset < 0, 'pasthistoryoffset'] = 0
pastHistory = pastHistory.groupby(['patientunitstayid', 'pasthistoryoffset'])['pasthistoryvalue'].apply(list).reset_index(name='pastHistory')
pastHistory = pastHistory[pastHistory.patientunitstayid.isin(cohort_icustay_id)]

In [24]:
pastHistory['itemname'] = 'pastHistory'
pastHistory.rename(index=str, columns={"pasthistoryoffset": "itemoffset",
                                       "labname": "itemname", "pastHistory": "itemvalue"}, inplace=True)
pastHistory = pastHistory[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]
pastHistory = pastHistory.reset_index(drop=True)

In [25]:
pastHistory.head(3)

In [26]:
print(pastHistory.patientunitstayid.nunique())
print(pastHistory.shape)

14984
(22392, 4)


### lab

In [27]:
def check(x):
    try:
        x = float(str(x).strip())
    except ValueError:
        try:
            x = float(re.findall(r'\d+', x)[0])
        except IndexError:
            x = np.nan
    return x

def check_itemvalue(df):
    df['itemvalue'] = df['itemvalue'].apply(lambda x: check(x))
    return df

In [28]:
lab = pd.read_csv(eicu + "lab.csv")

In [29]:
lab = lab[lab.labresultoffset >= 0]
lab.loc[lab.labmeasurenamesystem.isnull(), 'labmeasurenamesystem'] = lab.labmeasurenameinterface
lab = lab[['patientunitstayid', 'labresultoffset', 'labname', 'labresult', 'labresulttext', 'labmeasurenamesystem']]
lab = lab[(lab.labresult.notnull()) | (lab.labresulttext.notnull())]

In [30]:
lab['labresulttext'] = lab.labresulttext.map(lambda x: x.strip('<>%$'))
lab.loc[(lab.labresult.isnull()) & (lab.labresulttext.notnull()), 'labresult'] = lab['labresulttext']
lab.drop(columns=['labresulttext'], inplace=True)
lab.rename(index=str, columns={"labresultoffset": "itemoffset",
                               "labname": "itemname", "labresult": "itemvalue"}, inplace=True)

In [31]:
lab = check_itemvalue(lab)
lab = lab[lab.itemvalue.notnull()]

In [32]:
exclud_lab_var = ['Methemoglobin','Carboxyhemoglobin','CPK','Base Deficit','urinary specific gravity', 
                  'Oxyhemoglobin','serum osmolality','CPK-MB','prealbumin','HDL','BNP','urinary sodium','Fe',
                  'CPK-MB INDEX','urinary creatinine','LDL','Spontaneous Rate','TIBC','CRP','Vitamin B12',
                  'uric acid','PTT ratio','Tacrolimus-FK506','ESR','urinary osmolality','Fe/TIBC Ratio',
                  'cortisol','folate','free T4','protein - CSF','haptoglobin','Phenytoin','reticulocyte count',
                  'Device','Digoxin','ethanol','salicylate','Vent Other','CRP-hs','Gentamicin - trough',
                  'myoglobin','24 h urine protein','Gentamicin - random','serum ketones','Theophylline','T3',
                  'protein C','protein S','cd 4','Gentamicin - peak','Phenobarbital','T4','Tobramycin - random',
                  'Lithium','Carbamazepine','prolactin','ANF/ANA','Mode','Cyclosporin','Tobramycin - trough',
                  '24 h urine urea nitrogen','T3RU','Lidocaine','Amikacin - random','Tobramycin - peak',
                  'Amikacin - peak','HSV 1&2 IgG AB','Amikacin - trough','HIV 1&2 AB',
                  'Clostridium difficile toxin A+B','Legionella pneumophila Ab','Site','HSV 1&2 IgG AB titer',
                  'Patient s Comfort/Function (Pain) GOAL At Rest', "WBC's in urine", "WBC's in synovial fluid",
                  "WBC's in cerebrospinal fluid",  "WBC's in body fluid", "WBC's in peritoneal fluid", 
                  "WBC's in pleural fluid", "WBC's in pericardial fluid"]

lab = lab[~lab.itemname.isin(exclud_lab_var)]

In [33]:
lab_unit_df = lab[['itemname', 'labmeasurenamesystem']]
lab_unit_df = lab_unit_df.drop_duplicates()
lab_unit_df = lab_unit_df.groupby(['itemname'])['labmeasurenamesystem'].apply(list).reset_index(name='list_unit')
lab_unit_df['len'] = lab_unit_df.apply(lambda x: len(x['list_unit']), axis=1)
lab_unit_df = lab_unit_df[lab_unit_df.len > 1].reset_index(drop=True)

In [34]:
lab = lab[lab.patientunitstayid.isin(cohort_icustay_id)]
lab = lab.reset_index(drop=True)

In [35]:
lab.head(3)

In [36]:
print(lab.shape)
print(lab.patientunitstayid.nunique())

(3053582, 5)
15527


In [37]:
lab_unit_df.head(3)

,itemname,list_unit,len
0,Total CO2,"[mmol/L, nan, MMOL/L, mEq/L, MEQ/L, meq/L, mM/...",10
1,anion gap,"[mmol/L, nan, MMOLL, mEq/L, mmol/l, MMOL/L, mM...",8
2,pH,"[Units, nan, units, unit(s), pH units, Unit, p...",8


In [38]:
print(lab_unit_df.shape)

(3, 3)


### respiratoryCharting

In [39]:
respiratoryCharting = pd.read_csv(eicu + "respiratoryCharting.csv")

<ipython-input>:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  respiratoryCharting = pd.read_csv(eicu + "respiratoryCharting.csv")


In [40]:
respiratoryCharting = respiratoryCharting[respiratoryCharting.respchartoffset >= 0]
respiratoryCharting = respiratoryCharting[respiratoryCharting.respchartvaluelabel.notnull()]
respiratoryCharting = respiratoryCharting[respiratoryCharting.respchartvalue.notnull()]
respiratoryCharting = respiratoryCharting[['patientunitstayid', 'respchartoffset', 'respcharttypecat',
                                           'respchartvaluelabel', 'respchartvalue']]

In [41]:
respiratorySetting = respiratoryCharting[respiratoryCharting.respcharttypecat == 'respFlowSettings']
respiratoryData = respiratoryCharting[respiratoryCharting.respcharttypecat != 'respFlowSettings']

respiratorySetting.drop(columns=['respcharttypecat'], inplace=True)
respiratoryData.drop(columns=['respcharttypecat'], inplace=True)

<ipython-input>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  respiratorySetting.drop(columns=['respcharttypecat'], inplace=True)
<ipython-input>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  respiratoryData.drop(columns=['respcharttypecat'], inplace=True)


In [42]:
# respiratorySetting['respchartvaluelabel'].value_counts().rename_axis('unique_values').reset_index(name='counts')
low_measured_setting = ['Pressure Control', 'Pressure to Trigger PS', 'CPAP', '5. ARDS Eval (M or DNM)', 
                        '4. RSBI (RR/Vt)']

respiratorySetting = respiratorySetting[~respiratorySetting.respchartvaluelabel.isin(low_measured_setting)]

# respiratoryData['respchartvaluelabel'].value_counts().rename_axis('unique_values').reset_index(name='counts')
important_measured_var = ['RR (patient)', 'Total RR', 'Resp Rate Total', 'RR Spont','Spontaneous Respiratory Rate', 
                          'Exhaled MV', 'Exhaled TV (machine)', 'Exhaled TV (patient)', 'Exhaled Vt', 
                          'O2 Device', 'SaO2', 'Sedation outcome', 'Mean Airway Pressure',
                          'Plateau Pressure', 'Peak Insp. Pressure', 'FiO2', 'FIO2 (%)', 
                          'Set Fraction of Inspired Oxygen (FIO2)', 'O2 Percentage', 
                          'Spont TV', 'Tidal Volume Observed (VT)', 'Tidal Volume, Delivered', 'HR', 
                          'Oxygen Flow Rate', 'Peak Pressure', 'EtCO2', 'ETCO2', 'Inspiratory Flow Rate',
                          'Insp Flow (l/min)', 'Ventilator Type', 'Tracheostomy Type', 
                          'Ventilator Support Mode', 'Mechanical Ventilator Mode',
                          'Non-invasive Ventilation Mode']

respiratoryData = respiratoryData[respiratoryData.respchartvaluelabel.isin(important_measured_var)]

In [43]:
respiratorySetting = respiratorySetting[respiratorySetting.patientunitstayid.isin(cohort_icustay_id)]
respiratoryData = respiratoryData[respiratoryData.patientunitstayid.isin(cohort_icustay_id)]

In [44]:
respiratorySetting.rename(index=str, columns={"respchartoffset": "itemoffset", 
                                              "respchartvaluelabel":"itemname", 'respchartvalue': "itemvalue"},
                          inplace=True)
respiratorySetting = check_itemvalue(respiratorySetting)
respiratorySetting = respiratorySetting[respiratorySetting.itemvalue.notnull()]

respiratoryData.rename(index=str, columns={"respchartoffset": "itemoffset", 
                                           "respchartvaluelabel":"itemname", 'respchartvalue': "itemvalue"},
                        inplace=True)
respiratoryData = check_itemvalue(respiratoryData)
respiratoryData = respiratoryData[respiratoryData.itemvalue.notnull()]
respiratorySetting = respiratorySetting.reset_index(drop=True)
respiratoryData = respiratoryData.reset_index(drop=True)

In [45]:
respiratorySetting.head(3)

In [46]:
print(respiratorySetting.shape)
print(respiratorySetting.patientunitstayid.nunique())

(1424548, 4)
10157


In [47]:
respiratoryData.head(3)

In [48]:
print(respiratoryData.shape)
print(respiratoryData.patientunitstayid.nunique())

(1146398, 4)
8763


### vitalAperiodic

In [49]:
vitalAperiodic = []

for chunk in pd.read_csv(eicu + "vitalAperiodic.csv", chunksize=100000):
    chunk = chunk[chunk.patientunitstayid.isin(cohort_icustay_id)]
    chunk = chunk[chunk.observationoffset >= 0]
    chunk = chunk[['patientunitstayid', 'observationoffset', 'noninvasivesystolic', 
                   'noninvasivediastolic', 'noninvasivemean']]
    
    if chunk.shape[0] > 0:
        chunk.set_index(['patientunitstayid', 'observationoffset'], inplace=True)
        chunk = chunk.stack()
        chunk = chunk.reset_index()
        chunk.rename(index=str, columns={"observationoffset": "itemoffset",
                                         "level_2":"itemname", 0: "itemvalue"}, inplace=True)
        chunk = check_itemvalue(chunk)
        chunk = chunk[chunk.itemvalue.notnull()]
        vitalAperiodic.append(chunk)
        
vitalAperiodic = pd.concat(vitalAperiodic)

In [50]:
vitalAperiodic.head(3)

In [51]:
print(vitalAperiodic.shape)
print(vitalAperiodic.patientunitstayid.nunique())

(5277798, 4)
14859


### vitalPeriodic

In [52]:
vitalPeriodic = []

for chunk in pd.read_csv(eicu + "vitalPeriodic.csv", chunksize=100000):
    chunk = chunk[chunk.patientunitstayid.isin(cohort_icustay_id)]
    chunk = chunk[chunk.observationoffset >= 0]
    chunk = chunk[['patientunitstayid', 'observationoffset', 'temperature', 'heartrate', 'respiration', 'cvp',
                   'etco2', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic', 'padiastolic', 
                   'pamean', 'sao2', 'st1', 'st2', 'st3']]
    
    if chunk.shape[0] > 0:
        chunk.set_index(['patientunitstayid', 'observationoffset'], inplace=True)
        chunk = chunk.stack()
        chunk = chunk.reset_index()
        chunk.rename(index=str, columns={"observationoffset": "itemoffset",
                                         "level_2":"itemname", 0: "itemvalue"}, inplace=True)
        chunk = check_itemvalue(chunk)
        chunk = chunk[chunk.itemvalue.notnull()]
        vitalPeriodic.append(chunk)
        
vitalPeriodic = pd.concat(vitalPeriodic)

In [53]:
vitalPeriodic.head(3)

In [54]:
print(vitalPeriodic.shape)
print(vitalPeriodic.patientunitstayid.nunique())

(70405599, 4)
15157


### nurseCharting

In [55]:
nurseCharting = []

for chunk in pd.read_csv(eicu + "nurseCharting.csv", chunksize=100000):
    chunk = chunk[chunk.patientunitstayid.isin(cohort_icustay_id)]
    chunk = chunk[chunk.nursingchartoffset >= 0]
    chunk = chunk[['patientunitstayid','nursingchartoffset','nursingchartcelltypevallabel',
                   'nursingchartcelltypevalname', 'nursingchartvalue']]
    
    if chunk.shape[0] > 0:
        chunk.rename(index=str, columns={"nursingchartoffset": "itemoffset","nursingchartcelltypevalname":"itemname",
                                         "nursingchartcelltypevallabel" : "itemlabel",
                                         "nursingchartvalue": "itemvalue"}, inplace=True)
        chunk.loc[chunk['itemname']=='Value', 'itemname'] = chunk.itemlabel
        chunk.drop(columns=['itemlabel'], inplace=True)
        chunk = chunk[chunk.itemvalue.notnull()]
        nurseCharting.append(chunk)
        
nurseCharting = pd.concat(nurseCharting)
nurseCharting = nurseCharting.reset_index(drop=True)

In [56]:
GCS_nan = ['Unable to score due to medication']

nurseCharting.loc[(nurseCharting['itemname'] == 'GCS Total') & (nurseCharting['itemvalue'].isin(GCS_nan) ), 'itemvalue'] = np.nan

In [57]:
Motor_1 = ['1', '1-->(M1) none', 'Flaccid']
Motor_2 = ['2', '2-->(M2) extension to pain', 'Abnormal extension']
Motor_3 = ['3', '3-->(M3) flexion to pain', 'Abnormal flexion']
Motor_4 = ['4', '4-->(M4) withdraws from pain', 'Withdraws']
Motor_5 = ['5', '5-->(M5) localizes pain', 'Localizes to noxious stimuli']
Motor_6 = ['6', '6-->(M6) obeys commands', 'Obeys simple commands']

nurseCharting.loc[(nurseCharting['itemname'] == 'Best Motor Response') & (nurseCharting['itemvalue'].isin(Motor_1)), 'itemvalue'] = 1
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Motor Response') & (nurseCharting['itemvalue'].isin(Motor_2)), 'itemvalue'] = 2
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Motor Response') & (nurseCharting['itemvalue'].isin(Motor_3)), 'itemvalue'] = 3
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Motor Response') & (nurseCharting['itemvalue'].isin(Motor_4)), 'itemvalue'] = 4
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Motor Response') & (nurseCharting['itemvalue'].isin(Motor_5)), 'itemvalue'] = 5
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Motor Response') & (nurseCharting['itemvalue'].isin(Motor_6)), 'itemvalue'] = 6

In [58]:
Verbal_nan = ['Trached or intubated']
Verbal_1 = ['1', '1-->(V1) none', 'None', 'Clearly unresponsive']
Verbal_2 = ['2', '2-->(V2) incomprehensible speech', 'Incomprehensible sounds']
Verbal_3 = ['3', '3-->(V3) inappropriate words', 'Inappropriate words']
Verbal_4 = ['4', '4-->(V4) confused', 'Confused']
Verbal_5 = ['5', '5-->(V5) oriented', 'Oriented', 'Orientation/ability to communicate questionable',
            'Clearly oriented/can indicate needs']

nurseCharting.loc[(nurseCharting['itemname'] == 'Best Verbal Response') & (nurseCharting['itemvalue'].isin(Verbal_1)), 'itemvalue'] = 1
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Verbal Response') & (nurseCharting['itemvalue'].isin(Verbal_2)), 'itemvalue'] = 2
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Verbal Response') & (nurseCharting['itemvalue'].isin(Verbal_3)), 'itemvalue'] = 3
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Verbal Response') & (nurseCharting['itemvalue'].isin(Verbal_4)), 'itemvalue'] = 4
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Verbal Response') & (nurseCharting['itemvalue'].isin(Verbal_5)), 'itemvalue'] = 5
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Verbal Response') & (nurseCharting['itemvalue'].isin(Verbal_nan)), 'itemvalue'] = np.nan

In [59]:
Eye_1 = ['1', '1-->(E1) none']
Eye_2 = ['2', '2-->(E2) to pain']
Eye_3 = ['3', '3-->(E3) to speech']
Eye_4 = ['4', '4-->(E4) spontaneous']

nurseCharting.loc[(nurseCharting['itemname'] == 'Best Eye Response') & (nurseCharting['itemvalue'].isin(Eye_1)), 'itemvalue'] = 1
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Eye Response') & (nurseCharting['itemvalue'].isin(Eye_2)), 'itemvalue'] = 2
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Eye Response') & (nurseCharting['itemvalue'].isin(Eye_3)), 'itemvalue'] = 3
nurseCharting.loc[(nurseCharting['itemname'] == 'Best Eye Response') & (nurseCharting['itemvalue'].isin(Eye_4)), 'itemvalue'] = 4

In [60]:
nurseCharting.loc[(nurseCharting['itemname'] == 'Fall Risk') & (nurseCharting['itemvalue'] == 'Low'),    'itemvalue'] = 1
nurseCharting.loc[(nurseCharting['itemname'] == 'Fall Risk') & (nurseCharting['itemvalue'] == 'Medium'), 'itemvalue'] = 2
nurseCharting.loc[(nurseCharting['itemname'] == 'Fall Risk') & (nurseCharting['itemvalue'] == 'High'),   'itemvalue'] = 3

In [61]:
Delirium_0 = ['No', 'NO']
Delirium_1 = ['Yes', 'YES']
Delirium_nan = ['N/A']

nurseCharting.loc[(nurseCharting['itemname'] == 'Delirium Score') & (nurseCharting['itemvalue'].isin(Delirium_0)),  'itemvalue'] = 0
nurseCharting.loc[(nurseCharting['itemname'] == 'Delirium Score') & (nurseCharting['itemvalue'].isin(Delirium_1)),  'itemvalue'] = 1
nurseCharting.loc[(nurseCharting['itemname'] == 'Delirium Score') & (nurseCharting['itemvalue'].isin(Delirium_nan)),'itemvalue'] = np.nan

In [62]:
Symp_Delirium_0 = ['No']
Symp_Delirium_1 = ['Yes']

nurseCharting.loc[(nurseCharting['itemname'] == 'Symptoms of Delirium Present') & (nurseCharting['itemvalue'].isin(Symp_Delirium_0)),  'itemvalue'] = 0
nurseCharting.loc[(nurseCharting['itemname'] == 'Symptoms of Delirium Present') & (nurseCharting['itemvalue'].isin(Symp_Delirium_1)),  'itemvalue'] = 1

In [63]:
PainPresent_0 = ['No']
PainPresent_1 = ['Yes']
PainPresent_nan = ['Nonverbal', 'Asleep', 'Documentation undone']

nurseCharting.loc[(nurseCharting['itemname'] == 'Pain Present') & (nurseCharting['itemvalue'].isin(PainPresent_0)),  'itemvalue'] = 0
nurseCharting.loc[(nurseCharting['itemname'] == 'Pain Present') & (nurseCharting['itemvalue'].isin(PainPresent_1)),  'itemvalue'] = 1
nurseCharting.loc[(nurseCharting['itemname'] == 'Pain Present') & (nurseCharting['itemvalue'].isin(PainPresent_nan)),'itemvalue'] = np.nan

In [64]:
nurseCharting = nurseCharting[nurseCharting.itemvalue.notnull()]

In [65]:
nurseCharting.head(3)

In [66]:
print(nurseCharting.shape)
print(nurseCharting.patientunitstayid.nunique())

(14917073, 4)
15845


### intakeOutput

In [67]:
intakeOutput = pd.read_csv(eicu + "intakeOutput.csv")

In [68]:
intakeOutput = intakeOutput[['patientunitstayid', 'intakeoutputoffset', 'celllabel', 'cellvaluenumeric']]
intakeOutput = intakeOutput[intakeOutput.intakeoutputoffset >= 0]

In [69]:
Urine_output1 = ['Foley cath', 'Foley', 'foley', 'Urine Output-Foley', 'Urine Output-foley', 'Urine Output-FOLEY',
                 'FOLEY', 'Foley Cath', 'Foley catheter', 'Voided Amount', 'Urine, void:',
                 'Number of Incontinent Voids', 'unmeasured voids', 'R Nephrostomy', 'L Nephrostomy',
                 'Output (mL)-External Urethral Device Condom catheter', 'CONDOM CATHETER OUTPUT', 
                 'Condom Catheter', 'Output (mL)-Urinary Catheter Suprapubic catheter', 'Suprapubic Urine Output',
                 'Output (mL)-Suprapubic Catheter', 'Urinary Catheter Output: Suprapubic']

Urine_output2 = ['Urine','URINE CATHETER','Urine Count','Urine Occurrence','Urine, void:',
                 'Mixed Urine/Stool Volume','Urine Output-Foley','SN Urine Output(ml)',
                 'Urine Output (mL)-Urethral Catheter','Urine Output (mL)-Urethral Catheter ',
                 'Urine Output-foley','Urine Output (mL)-Urethral Catheter  ','Urine Output-FOLEY',
                 'Urine Output-Urine Output','Suprapubic Urine Output','urine count',
                 'Urine Output-RIGHT Nephrouretero Stent Urine Output','urine incontinence',
                 'Urine Output-Nephrostomy','Urine Incontinence','incontinent urine','Urine, L neph:','OR urine']

Propofol = ['propofol','Volume (mL) Propofol ','Volume (ml) Propofol','Propofol','PROPOFOL','Propofol v2-IN',
            'PROpofol (ml):','Propofol drip','Propofol gtt','PROPOFOL DRIP',
            'Volume (mL)-propofol (DIPRIVAN) IV bolus','Propofol ','PROPOFOL GTT']

Fentanyl = ['Volume (mL) Fentanyl','fentaNYL','Volume (ml) Fentanyl','Fentanyl','fentanyl',
            'Total Amount (mcg)-fentaNYL PCA 50 mcg/mL','FENTANYL','Fentanyl-IN','Fentanyl (NS)-IN',
            'Fentanyl PCA Volume','FENTANYL GTT','Fentanyl ','Fentanyl drip','Fentanyl gtt ','Fentanyl IV',
            'Volume Infused (ml)-fentaNYL PCA 50 mcg/mL','FENTANYL ','Fentanal','Fentanyl gtt','fentanyl drip']

Insulin = ['Volume (ml) Insulin','insulin regular','Volume (mL) Insulin ','Insulin','INSULIN','insulin',
           'Insulin Drip (NS)-IN','Insulin drip','Insulin Drip','TPN with insulin','INSULIN DRIP','INSULIN GTT',
           'Insulin Gtt','INsulin IV']

Heparin = ['Volume (mL) Heparin','heparin','Volume (ml) Heparin','Heparin','Heparin (NS 0.45%)-IN','HEPARIN',
           'HEPARIN DRIP','Heparin Drip','Heparin (D5W)-IN','Heparin gtt','heparin drip','heparin ','Heparin drip',
           'HEPARIN DRIP ','Heparin 100 Units-IN', 'Heparin ']

Midazolam = ['Volume (mL) Midazolam','Volume (ml) Midazolam','midazolam','Midazolam (NS)-IN',
             'Midazolam-IN','MIDAZOLAM']

Dexmedetomidine = ['dexmedetomidine','Volume (ml) Dexmedetomidine','Volume (mL) Dexmedetomidine',
                   'DEXmedetomidine (ml):','DEXMEDETOMIDINE']

Vassopressin = ['VASSOPRESSIN', 'vassopressin']

Albumin = ['Volume (mL)-albumin human 5 % injection 12.5 g','5% Albumin',
           'Volume (mL)-albumin human 5 % injection 25 g','Albumin 5%',
           'Volume (mL)-albumin human 25 % injection 25 g',
           'Volume (mL)-albumin human (SPA) 25 % injection 12.5 g','25% Albumin',
           'Volume (mL)-albumin human (SPA) 25 % injection 25 g','Albumin 25%',
           'Volume (mL)-albumin human 5 % solution 250 mL','25% Albumin 50 ml',
           '25% Albumin 100 ml','5% Albumin 500 ml','Volume (mL)-albumin human 5 % injection',
           'Volume (mL)-ALBUMIN HUMAN 5 % IV SOLN Pyxis Override','Albumin Intake',
           'Volume (mL)-albumin human 25 % injection 12.5 g',
           'Volume (mL)-albumin human 5 % solution 12.5 g',
           'Volume (mL)-albumin human 5 % injection 12.5-25 g',
           'Volume (mL)-albumin human 25 % solution 25 g','albumin']

Ceftriaxone = ['Volume (mL)-cefTRIAXone (ROCEPHIN) 1 g in dextrose 5 % 50 mL IVPB',
               'Volume (mL)-cefTRIAXone (ROCEPHIN) 2 g in sodium chloride 0.9 % 100 mL IVPB',
               'Volume (mL)-cefTRIAXone (ROCEPHIN) 1,000 mg in sodium chloride 0.9 % 100 mL IVPB',
               'Volume (mL)-cefTRIAXone (ROCEPHIN) 2 g in dextrose 5 % 50 mL IVPB',
               'Volume (mL)-cefTRIAXone (ROCEPHIN) 1 g in sodium chloride 0.9 % 100 mL IVPB',
               'Volume (mL)-cefTRIAXone (ROCEPHIN) 2,000 mg in sodium chloride 0.9 % 100 mL IVPB']

Cefazolin = ['Volume (mL)-ceFAZolin 2,000 mg in sodium chloride 0.9% 100 mL IVPB',
             'Volume (mL)-ceFAZolin (ANCEF) in sod chl 2 g IVPB premix',
             'Volume (mL)-ceFAZolin (ANCEF) IVPB 2 g/50 mL premix',
             'Volume (mL)-ceFAZolin (ANCEF) IVPB 1 g/50 mL premix',
             'Volume (mL)-ceFAZolin (ANCEF) 1,000 mg in sodium chloride 0.9 % 100 mL IVPB',
             'Volume (mL)-ceFAZolin (ANCEF) in d5w 2 g premix',
             'Volume (mL)-ceFAZolin (ANCEF) in sod chl 3 g IVPB premix']

Cefepime = ['Volume (mL)-cefepime (MAXIPIME) 1 g in sodium chloride 0.9 % 100 mL IVPB',
            'Volume (mL)-cefepime (MAXIPIME) 2 g in sodium chloride 0.9 % 100 mL IVPB',
            'Volume (mL)-ceFEPIme (MAXIPIME) 1 g in dextrose 5 % 50 mL IVPB',
            'Volume (mL)-ceFEPIme (MAXIPIME) 1,000 mg in sodium chloride 0.9 % 100 mL IVPB',
            'Volume (mL)-ceFEPIme (MAXIPIME) 2 g in dextrose 5 % 50 mL IVPB',
            'Volume (mL)-ceFEPIme (MAXIPIME) 2,000 mg in sodium chloride 0.9 % 100 mL IVPB',
            'Cefepime']

Ceftazidime = ['Volume (mL)-cefTAZidime (FORTAZ) 1 g in dextrose 5 % 50 mL IVPB',
               'Volume (mL)-cefTAZidime (FORTAZ) 2 g in dextrose 5 % 50 mL IVPB']

Vancomycin = ['Volume (mL)-vancomycin (VANCOCIN) IVPB 1000 mg/200 mL premix',
              'Volume (mL)-vancomycin (VANCOCIN) 1,500 mg in dextrose 5 % 250 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 1,250 mg in dextrose 5 % 250 mL IVPB',
              'Volume (mL)-vancomycin in dextrose premix 1 g',
              'Volume (mL)-vancomycin (VANCOCIN) IVPB 750 mg/150 ml premix',
              'Volume (mL)-vancomycin (VANCOCIN) 1,250 mg in sodium chloride 0.9 % 250 mL IVPB',
              'Volume (mL)-vancomycin 1,250 mg in sodium chloride 0.9% 250 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 1,000 mg in sodium chloride 0.9 % 250 mL IVPB',
              'Volume (mL)-vancomycin in dextrose premix 1,000 mg',
              'Volume (mL)-vancomycin (VANCOCIN) 500 mg in sodium chloride 0.9 % 100 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 1,500 mg in sodium chloride 0.9 % 500 mL IVPB',
              'vancomycin', 'Vancomycin', 'VANCOMYCIN',
              'Volume (mL)-vancomycin (VANCOCIN) 750 mg in dextrose 5 % 250 mL IVPB',
              'Volume (mL)-vancomycin 1,500 mg in sodium chloride 0.9% 250 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 1,750 mg in dextrose 5 % 500 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 1,500 mg in sodium chloride 0.9 % 250 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 750 mg in sodium chloride 0.9 % 250 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 1,750 mg in sodium chloride 0.9 % 500 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 2,000 mg in sodium chloride 0.9 % 500 mL IVPB',
              'Volume (mL)-vancomycin (VANCOCIN) 750 mg in sodium chloride 0.9 % 150 mL IVPB']

Clindamycin = ['Volume (mL)-clindamycin (CLEOCIN) IVPB 600 mg',
               'Volume (mL)-clindamycin (CLEOCIN) IVPB 900 mg',
               'Volume (mL)-clindamycin 600 mg in sodium chloride 0.9% 50 mL IVPB',
               'Volume (mL)-clindamycin 900 mg in sodium chloride 0.9% 50 mL IVPB']

Metronidazole = ['Volume (mL)-metroNIDAZOLE (FLAGYL) IV 500 mg',
                 'Volume (mL)-metroNIDAZOLE (FLAGYL) IVPB 500 mg']

Meropenem = ['Volume (mL)-meropenem (MERREM) 500 mg in sodium chloride 0.9 % 100 mL IVPB',
             'Volume (mL)-meropenem (MERREM) 1 g in sodium chloride 0.9 % 100 mL IVPB',
             'Meropenem', 'meropenem',
             'Volume (mL)-meropenem (MERREM) 2 g in sodium chloride 0.9 % 100 mL IVPB']

Acyclovir = ['Acyclovir']

Azithromycin = ['Volume (mL)-azithromycin (ZITHROMAX) 500 mg in sodium chloride 0.9 % 250 mL IVPB',
                'Volume (mL)-azithromycin (ZITHROMAX) 500 mg in dextrose 5 % 250 mL IVPB']

Levofloxacin = ['Volume (mL)-levofloxacin (LEVAQUIN) IVPB 750 mg',
                'Volume (mL)-levofloxacin (LEVAQUIN) in dextrose 5% premix IVPB 750 mg',
                'Volume (mL)-levofloxacin (LEVAQUIN) IVPB 500 mg',
                'Volume (mL)-levofloxacin (LEVAQUIN) in dextrose 5% premix IVPB 500 mg']

Micafungin = ['Volume (mL)-micafungin (MYCAMINE) 100 mg in sodium chloride 0.9 % 100 mL IVPB',
              'Volume (mL)-micafungin 100 mg in sodium chloride 0.9 % 100 mL IVPB',
              'Volume (mL)-Micafungin Sodium 100 mg in sodium chloride 0.9 % 100 mL IVPB']

Fluconazole = ['Volume (mL)-fluconazole (DIFLUCAN) IVPB 200 mg',
               'Volume (mL)-fluconazole (DIFLUCAN) IVPB 400 mg',
               'Volume (mL)-fluconazole (DIFLUCAN) IVPB 100 mg']

SodiumChloride = ['Volume (mL)-0.9 %  sodium chloride infusion','Volume (mL)-0.45 % sodium chloride infusion',
                  'Volume (mL)-dextrose 5 %-0.45 % sodium chloride infusion', 
                  'Volume (mL)-dextrose 5 %-0.9 % sodium chloride infusion',
                  'Volume (mL)-dextrose 5 % and 0.2% sodium chloride infusion', 
                  'Volume (mL)-dextrose 5 % and 0.2 % sodium chloride infusion',
                  'Volume (mL)-0.9 % sodium chloride solution', 'Volume (mL)-0.45 % sodium chloride solution']

Thiamine = ['Volume (mL)-ciprofloxacin (CIPRO) IVPB 400 mg',
            'Volume (mL)-ciprofloxacin (CIPRO) 400 mg in dextrose 5% 200 mL IVPB']

Dobutamine = ['Volume (ml) Dobutamine', 'DOBUTamine','Volume (mL) Dobutamine','Dobutamine','DOBUTAMINE',
              'Dobutamine-IN','Dobutamine (NS)-IN', 'DOButamine (ml):','dobutamine','DOBUTAMINE ']

Milrinone = ['Volume (mL) Milrinone ','Volume (ml) Milrinone','milrinone','Milrinone','Milrinone-IN',
             'MILrinone (ml):']

Fluids = ['PO fluids', 'PO Fluids','PO FLUIDS','PO po fluids', 'PO Fluids ','PO PO Fluids','PO PO FLUIDS',
          'PO PO fluids','PO oral fluids','Enteral Fluids','PO fluids ']

OralIntake = ['Oral Intake', 'Oral Intake Amount']

PO = ['P.O.', 'P.O. Intake']

Weight = ['Bodyweight (kg)', 'Bodyweight (lb)']

IVPB = ['IVPB', 'IVPB Volume (ml)']

Stool = ['Stool', 'Stool Occurrence', 'Liquid Stool', 'Stool Volume', 'Output, Stool Amount',
         'Liquid stool:', 'Stool Output (mL)-Ileostomy ileostomy']

Crystalloids = ['Crystalloids']

NSIVF = ['NS IVF'] 

Norepinephrine = ['Volume (mL) Norepinephrine', 'norepinephrine', 'Volume (ml) Norepinephrine',
                  'Norepinephrine V3-IN', 'NOREPINEPHRINE', 'Norepinephrine', 'CV Norepinephrine 8mg V2-IN']

Amiodarone = ['Volume (mL) Amiodarone', 'Volume (ml) Amiodarone', 'amiodarone', 'Amiodarone', 'Amiodarone-IN',
              'AMIODARONE', 'Amiodarone drip', 'amiodarone drip', 'Amiodarone gtt', 'Amiodarone (0.45% NS)-IN']

Phenylephrine = ['Volume (mL) Phenylephrine ', 'Volume (ml) Phenylephrine', 'phenylephrine',
                 'Phenylephrine (NS)-IN', 'Phenylephrine-IN', 'Phenylephrine', 'PHENYLEPHRINE']

Epinephrine = [ 'Volume (ml) Epinephrine', 'Volume (mL) Epinephrine', 'EPINEPHrine', 'Epinephrine',
               'EPInephrine (ml):', 'EPINEPHRINE', 'Epinephrine drip', 'epinephrine', 'Epinephrine (NS)-IN']

Nicardipine = ['niCARdipine', 'Volume (mL) Nicardipine', 'Volume (ml) Nicardipine', 'Volume (ml)  Nicardipine',
               'Nicardipine (NS)-IN', 'Nicardipine-IN', 'NICARDIPINE', 'Nicardipine', 'nicardipine']

Pantoprazole = ['PANTOPRAZOLE', 'Volume (mL) Pantoprazole', 'pantoprazole', 'Volume (ml) Pantoprazole',
                'Pantoprazole (NS)-IN', 'Volume (mL)-pantoprazole (PROTONIX) injection 40 mg']

Diltiazem = ['Volume (mL) Diltiazem', 'diltiazem', 'Volume (ml) Diltiazem', 'Diltiazem-IN', 'DILTIAZEM',
             'Diltiazem (0.45% NS)-IN']

Nitroglycerin = ['Volume (mL) Nitroglycerin ', 'nitroglycerin', 'Volume (ml) Nitroglycerin', 'NITROGLYCERIN',
                 'Nitroglycerin-IN', 'Nitroglycerin', 'CV Nitroglycerin-IN', 'NITroglycerine (ml):']

In [70]:
intakeOutput = intakeOutput[intakeOutput.celllabel.isin(Urine_output1)  | intakeOutput.celllabel.isin(Urine_output2)| 
                            intakeOutput.celllabel.isin(Propofol)       | intakeOutput.celllabel.isin(Fentanyl)     | 
                            intakeOutput.celllabel.isin(Insulin)        | intakeOutput.celllabel.isin(Heparin)      | 
                            intakeOutput.celllabel.isin(Midazolam)      | intakeOutput.celllabel.isin(Dexmedetomidine)| 
                            intakeOutput.celllabel.isin(Vassopressin)   | intakeOutput.celllabel.isin(Albumin)      |
                            intakeOutput.celllabel.isin(Ceftriaxone)    | intakeOutput.celllabel.isin(Cefazolin)    |
                            intakeOutput.celllabel.isin(Cefepime)       | intakeOutput.celllabel.isin(Ceftazidime)  |
                            intakeOutput.celllabel.isin(Vancomycin)     | intakeOutput.celllabel.isin(Clindamycin)  |
                            intakeOutput.celllabel.isin(Metronidazole)  | intakeOutput.celllabel.isin(Meropenem)    |
                            intakeOutput.celllabel.isin(Acyclovir)      | intakeOutput.celllabel.isin(Azithromycin) |
                            intakeOutput.celllabel.isin(Levofloxacin)   | intakeOutput.celllabel.isin(Micafungin)   |
                            intakeOutput.celllabel.isin(Fluconazole)    | intakeOutput.celllabel.isin(Thiamine)     |
                            intakeOutput.celllabel.isin(Dobutamine)     | intakeOutput.celllabel.isin(Milrinone)    |
                            intakeOutput.celllabel.isin(Fluids)         | intakeOutput.celllabel.isin(OralIntake)   |
                            intakeOutput.celllabel.isin(PO)             | intakeOutput.celllabel.isin(Weight)       |
                            intakeOutput.celllabel.isin(SodiumChloride) | intakeOutput.celllabel.isin(IVPB)         |
                            intakeOutput.celllabel.isin(Stool)          | intakeOutput.celllabel.isin(Crystalloids) |
                            intakeOutput.celllabel.isin(NSIVF)          | intakeOutput.celllabel.isin(Norepinephrine)|
                            intakeOutput.celllabel.isin(Amiodarone)     | intakeOutput.celllabel.isin(Phenylephrine)|
                            intakeOutput.celllabel.isin(Epinephrine)    | intakeOutput.celllabel.isin(Nicardipine)  |
                            intakeOutput.celllabel.isin(Pantoprazole)   | intakeOutput.celllabel.isin(Diltiazem)    |
                            intakeOutput.celllabel.isin(Nitroglycerin)]

In [71]:
intakeOutput.rename(index=str, columns={"intakeoutputoffset" : "itemoffset",
                                        "celllabel" : "itemname", "cellvaluenumeric" : "itemvalue"}, inplace=True)
intakeOutput = check_itemvalue(intakeOutput)
intakeOutput = intakeOutput[intakeOutput.itemvalue.notnull()]

In [72]:
intakeOutput.loc[intakeOutput.itemname.isin(Urine_output1)  ,  'itemname'] = 'Urine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Urine_output2)  ,  'itemname'] = 'Urine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Propofol)  ,       'itemname'] = 'Propofol_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Fentanyl)  ,       'itemname'] = 'Fentanyl_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Insulin)  ,        'itemname'] = 'Insulin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Heparin)  ,        'itemname'] = 'Heparin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Midazolam)  ,      'itemname'] = 'Midazolam_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Dexmedetomidine) , 'itemname'] = 'Dexmedetomidine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Vassopressin)  ,   'itemname'] = 'Vassopressin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Albumin)  ,        'itemname'] = 'Albumin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Ceftriaxone)  ,    'itemname'] = 'Ceftriaxone_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Cefazolin)  ,      'itemname'] = 'Cefazolin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Cefepime)  ,       'itemname'] = 'Cefepime_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Ceftazidime)  ,    'itemname'] = 'Ceftazidime_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Vancomycin)  ,     'itemname'] = 'Vancomycin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Clindamycin)  ,    'itemname'] = 'Clindamycin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Metronidazole)  ,  'itemname'] = 'Metronidazole_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Meropenem)  ,      'itemname'] = 'Meropenem_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Acyclovir)  ,      'itemname'] = 'Acyclovir_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Azithromycin)  ,   'itemname'] = 'Azithromycin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Levofloxacin)  ,   'itemname'] = 'Levofloxacin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Micafungin)  ,     'itemname'] = 'Micafungin_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Fluconazole)  ,    'itemname'] = 'Fluconazole_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Thiamine)  ,       'itemname'] = 'Thiamine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Dobutamine)  ,     'itemname'] = 'Dobutamine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Milrinone)  ,      'itemname'] = 'Milrinone_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Fluids)  ,         'itemname'] = 'Fluids_IO'
intakeOutput.loc[intakeOutput.itemname.isin(OralIntake)  ,     'itemname'] = 'OralIntake_IO'
intakeOutput.loc[intakeOutput.itemname.isin(PO)  ,             'itemname'] = 'P.O._IO'
intakeOutput.loc[intakeOutput.itemname.isin(SodiumChloride)  , 'itemname'] = 'SodiumChloride_IO'
intakeOutput.loc[intakeOutput.itemname.isin(IVPB)  ,           'itemname'] = 'IVPB_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Stool)  ,          'itemname'] = 'Stool_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Crystalloids)  ,   'itemname'] = 'Crystalloids_IO'
intakeOutput.loc[intakeOutput.itemname.isin(NSIVF)  ,          'itemname'] = 'NSIVF_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Norepinephrine)  , 'itemname'] = 'Norepinephrine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Amiodarone)  ,     'itemname'] = 'Amiodarone_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Phenylephrine)  ,  'itemname'] = 'Phenylephrine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Epinephrine)  ,    'itemname'] = 'Epinephrine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Nicardipine)  ,    'itemname'] = 'Nicardipine_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Pantoprazole)  ,   'itemname'] = 'Pantoprazole_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Diltiazem)  ,      'itemname'] = 'Diltiazem_IO'
intakeOutput.loc[intakeOutput.itemname.isin(Nitroglycerin)  ,  'itemname'] = 'Nitroglycerin_IO'

In [73]:
intakeOutput = intakeOutput[intakeOutput.patientunitstayid.isin(cohort_icustay_id)]
intakeOutput = intakeOutput.reset_index(drop=True)

In [74]:
intakeOutput.head(3)

In [75]:
print(intakeOutput.shape)
print(intakeOutput.patientunitstayid.nunique())

(882725, 4)
15260


### infusionDrug

In [76]:
infusionDrug = pd.read_csv(eicu + "infusionDrug.csv")

<ipython-input>:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  infusionDrug = pd.read_csv(eicu + "infusionDrug.csv")


In [77]:
infusionDrug = infusionDrug[infusionDrug.infusionoffset >= 0]
infusionDrug = infusionDrug[infusionDrug.drugname.notnull()]
infusionDrug = infusionDrug[(infusionDrug.drugrate.notnull()) | (infusionDrug.drugamount.notnull())]
infusionDrug = infusionDrug[['patientunitstayid', 'infusionoffset', 'drugname']]

In [78]:
Fentanyl = ['Fentanyl (ml/hr)', 'Fentanyl (mcg/hr)', 'Fentanyl ()', 'Fentanyl (mcg/kg/hr)', 
            'Fentanyl (mcg/kg/min)', 'FentaNYL (Sublimaze) 2500 mcg Sodium Chloride 0.9% 250 ml  Premix (mcg/hr)',
            'Fentanyl PCA (mcg/hr)', 'Fentanyl (mg/hr)', 'fentanyl (mcg/hr)', 'Fentanyl (Unknown)', 
            'Fentanyl/Bupivi Epidural (ml/hr)', 'epidural; fentanyl/lido/duramorph (ml/hr)',
            'Bupivacaine 0.5%/fentanyl 1000mcg (ml/hr)', 'fentanyl/bupivicaine (ml/hr)', 
            'Epidural-Bupivicaine/Fentanyl (ml/hr)', 'Fentanyl625 mcg/Bupivicaine 312.5 mg (ml/hr)', 
            'fentanyl/bupivacaine (ml/hr)', 'Fentanyl (mcg/min)', 'fentanyl (mcg/kg/min)',
            'epidural :fentanyl/duramorph/bupivicaine (ml/hr)', 'Fentanyl/Bupivacaine/Dilaudid (ml/hr)',
            'FENTANYL PCA (mcg/hr)', 'Fentanyl/Bupivicaine PCEA (ml/hr)', 
            'FentaNYL (Sublimaze) 2500 mcg Sodium Chloride 0.9% 500 ml  Premix (mcg/hr)',
            'fentanyl epidural (ml/hr)', 'Fentanyl/Bup Epidural (ml/hr)', 'fentanyl  (mcg/hr)', 
            'Epidural Fentanyl (ml/hr)', 'fentanyl epi (ml/hr)', 'fentanyl/epidural (ml/hr)', 
            'Fentanyl epidural (ml/hr)', 'fentanyl PCA (mcg/hr)', 'Fentanyl PCA (cont) (ml/hr)', 
            'fentaNYL PCA 100 ml ()', 'Epidural/fentanyl (ml/hr)']

Propofol = ['Propofol (ml/hr)', 'Propofol (mcg/kg/min)', 'Propofol ()', 
            'Propofol (Diprivan) 1000 mg  100 ml  Premix (mcg/kg/min)', 'Propofol (mg/kg/min)', 
            'propofol (mcg/kg/min)', 'Propofol (mcg/kg/hr)', 'PROPOFOL (mcg/kg/min)', 'Propofol (mcg/min)', 
            'Propofol (mg/hr)', 'Propofol (Unknown)', 'Propofol (mcg/hr)', 
            'Propofol (Diprivan) 1000 mg  130 ml  Premix (mcg/kg/min)', 'NS w/ propofol (ml/hr)', 
            'Propofol IV Emulsion 100 ml (ml/hr)', 'Propofol (mg/kg/hr)', 'propofol (ml/hr)', 
            'Propofol (mg/min)', 'propofol (mcg/min)', 
            'Propofol (Diprivan) 1000 mg Sterile Water (SWFI) 100 ml  Premix (mcg/kg/min)']

Norepinephrine = ['Norepinephrine (ml/hr)', 'Norepinephrine (mcg/min)', 'Norepinephrine (mcg/kg/min)', 
                  'Norepinephrine ()', 'norepinephrine Volume (ml) (ml/hr)', 
                  'Norepinephrine STD 4 mg Dextrose 5% 250 ml (mcg/min)', 
                  'Norepinephrine MAX 32 mg Dextrose 5% 250 ml (mcg/min)', 'Norepinephrine (mcg/kg/hr)', 
                  'Norepinephrine (mg/hr)', 'Norepinephrine (mcg/hr)', 
                  'Norepinephrine MAX 32 mg Dextrose 5% 500 ml (mcg/min)', 
                  'Norepinephrine STD 8 mg Dextrose 5% 500 ml (mcg/min)', 'Norepinephrine (mg/kg/min)', 
                  'Norepinephrine STD 4 mg Dextrose 5% 500 ml (mcg/min)', 
                  'Norepinephrine STD 8 mg Dextrose 5% 250 ml (mcg/min)', 'Norepinephrine (mg/min)',
                  'Norepinephrine (Unknown)', 'Norepinephrine STD 32 mg Dextrose 5% 282 ml (mcg/min)',
                  'Norepinephrine (units/min)', 'Norepinephrine', 'norepinephrine Volume (ml)',
                  'Norepinephrine STD 32 mg Dextrose 5% 500 ml (mcg/min)',
                  'Levophed (ml/hr)', 'levophed (mcg/min)', 'Levophed (mg/hr)', 'levophed (ml/hr)',
                  'Levophed (mcg/kg/min)', 'NSS w/ levo/vaso (ml/hr)', 'levophed  (mcg/min)',
                  'NSS with LEVO (ml/hr)', 'Levophed (mcg/min)']

Insulin = ['Insulin (units/hr)', 'Insulin (ml/hr)', 'Insulin ()', 
           'Insulin 250 Units Sodium Chloride 0.9% 250 ml (units/hr)', 'Insulin Scale A (units/hr)', 
           'Insulin (units/kg/hr)', 'Insulin Scale B (units/hr)', 'Insulin- Scale A (units/hr)', 
           'insulin (units/hr)', 'Insulin (mg/hr)', 'Insulin (Scale A) (units/hr)', 'TPN w/insulin (ml/hr)', 
           'Insulin (Scale B) (units/hr)', 'Insulin (scaleB) (units/hr)', 'insulin (ml/hr)', 
           'Insuline Scale B (units/hr)', 'Insulin Scale A  (units/hr)', 'Insulin- Scale B (units/hr)', 
           'REGULAR INSULIN (units/hr)', 'Scale A Insulin (units/hr)', 'Insulin (Regular) (ml/hr)',
           'Insulin Regular (ml/hr)', 'Insulin (Unknown)', 'Insulin scale A (units/hr)', 
           'Humulin R Insulin (ml/hr)', 'Insulin gtt (units/hr)', 'Insulin B (units/hr)', 
           'Insulin Regular (units/hr)', 'Bolus Insulin (units/hr)', 'insulin drip (units/hr)', 
           'bolus insulin (units/hr)', 'D30W/85 meq KCL/60 units insulin (ml/hr)', 'Insulin (mcg/min)', 
           'Bolus insulin (units/hr)', 'Insulin (mcg/kg/hr)', 'Insulin (mg/min)']

Midazolam = ['Midazolam (mg/hr)', 'Midazolam (ml/hr)', 'Midazolam ()',
             'Midazolam (Versed) 100 mg Sodium Chloride 0.9% 100 ml (mg/hr)', 'Midazolam (mcg/kg/min)',
             'Midazolam (mg/kg/hr)', 'Midazolam (Versed) 100 mg Sodium Chloride 0.9% 350 ml (mg/hr)', 
             'Midazolam (mcg/hr)', 'Midazolam (mcg/kg/hr)']

Heparin = ['Heparin (ml/hr)', 'Heparin (units/hr)', 'Heparin ()', 'heparin (units/hr)', 
           'Volume (ml) Heparin-heparin 25,000 units in dextrose 500 mL infusion (ml/hr)',
           'Volume (ml) Heparin-heparin 25,000 units in 0.45 % sodium chloride 500 mL infusion (ml/hr)', 
           'Heparin (units/kg/hr)', 'Heparin 25000 Units Dextrose 5% 500 ml  Premix (units/kg/hr)', 
           'Volume (ml) Heparin-heparin infusion 2 units/mL in 0.9% sodium chloride (ARTERIAL LINE) (ml/hr)', 
           'HEPARIN (units/hr)', 'NSS carrier heparin (ml/hr)', 'Heparin (Unknown)', 
           'Heparin 25000 Units Dextrose 5% 500 ml  Premix (units/hr)', 'Heparin-EKOS (units/hr)', 
           'Heparin via sheath (units/hr)', 'HEPARIN #2 (units/hr)', 'Heparin (mcg/kg/hr)', 
           'Heparin 25000 Units Dextrose 5% 950 ml  Premix (units/kg/hr)', 'Heparin (mcg/kg/min)', 
           'Heparin 8000u/1L NS (ml/hr)', 'Heparin/Femoral Sheath   (units/hr)', 'S-Heparin (units/hr)', 
           'Left  Heparin (units/hr)', 'Heparin 25,000 Unit/D5w 250 ml (ml/hr)', 'Heparin', 'Hepain (ml/hr)',
           'Volume (ml) Heparin-heparin 25,000 units in 0.45 % sodium chloride 500 mL infusion', 
           'Volume (ml) Heparin-heparin 25,000 units in dextrose 500 mL infusion',
           'Volume (ml) Heparin-heparin infusion 2 units/mL in 0.9% sodium chloride (ARTERIAL LINE)']

Dexmedetomidine = ['Dexmedetomidine (ml/hr)', 'Dexmedetomidine (mcg/kg/hr)', 
                   'Dexmedetomidine (mcg/kg/hr) (mcg/kg/hr)', 
                   'Dexmedetomidine(Precedex) 400 mcg Sodium Chloride 0.9% 100 ml (mcg/kg/hr)', 
                   'Dexmedetomidine Inj 400 Mcg in Sodium Chloride 0.9% 100 ml (mcg/kg/hr)', 
                   'Dexmedetomidine Inj 400 Mcg in Dextrose 5% 100 ml (mcg/kg/hr)', 
                   'Dexmedetomidine (mcg/min)', 'Dexmedetomidine (mcg/hr)', 'Dexmedetomidine (mg/hr)',
                   'Dexmedetomidine (mcg/kg/min)', 'Dexmedetomidine (mg/kg/hr)', 
                   'Dexmedetomidine Inj 400 Mcg in Sodium Chloride 0.9% 100 ml (ml/hr)', 
                   'Dexmedetomidine Inj 400 Mcg in Sodium Chloride 0.9% 100 ml ()', 
                   'Dexmedetomidine (mg/kg/min)', 'Dexmedetomidine (mcg/kg/hr) ()', 
                   'Dexmedetomidine Inj 400 Mcg in Dextrose 5% 100 ml ()', 
                   'Dexmedetomidine Inj 400 Mcg in Dextrose 5% 100 ml (mcg/min)', 
                   'Dexmedetomidine Inj 400 Mcg in Sodium Chloride 0.9% 100 ml (mcg/kg/min)', 
                   'Dexmedetomidine Inj 400 Mcg in Sodium Chloride 0.9% 100 ml (mcg/min)', 
                   'Dexmedetomidine Inj 400 Mcg in Dextrose 5% 100 ml (ml/hr)','Dexmedetomidine (units/min)']

Amiodarone = ['Amiodarone (ml/hr)', 'Amiodarone (mg/min)', 'Amiodarone ()', 'Amiodarone (mg/min) (mg/min)', 
              'Amiodarone (maintenance) 900 mg Dextrose 5% 500 ml (mg/min)', 'Amiodarone (mg/hr)', 
              'Amiodarone (mcg/min) (mcg/min)', 'Amiodarone (load) 150 mg Dextrose 5% 100 ml (mg/min)', 
              'Amiodarone (mcg/min)', 'amiodarone (mg/min)', 'Amiodarone (Unknown)', 'amiodarone bolus (mg/min)',
              'Amiodarone Bolus (mg/min)', 'Amiodarone Inj 450 mg in Dextrose 5% (Aviva) 250 ml ()',
              'Amiodarone (mcg/kg/min)']

Vasopressin = ['Vasopressin (ml/hr)', 'Vasopressin (units/min)', 'Vasopressin ()', 'Vasopressin (units/hr)', 
               'Vasopressin 40 Units Sodium Chloride 0.9% 100 ml (units/min)', 'vasopressin (units/min)',
               'VAsopressin (units/min)', 'Vasopressin (mg/min)', 'Vasopressin (Unknown)', 'Vasopressin (mcg/min)', 
               'Vasopressin 40 Units Sodium Chloride 0.9% 100 ml (units/hr)',
               'Vasopressin 40 Units Sodium Chloride 0.9% 100 ml (units/kg/hr)',
               'Vasopressin 20 Units Sodium Chloride 0.9% 100 ml (units/hr)', 
               'Vasopressin 40 Units Sodium Chloride 0.9% 100 ml (Unknown)', 'Vasopressin (mcg/kg/min)', 
               'Vasopressin 20 Units Sodium Chloride 0.9% 250 ml (units/hr)', 
               'Vasopressin 40 Units Sodium Chloride 0.9% 200 ml (units/min)', 'Vasopressin (mg/hr)',
               'Vasopressin (units/kg/min)', 'vasopressin (ml/hr)', 'Vasopressin']

Phenylephrine = ['Phenylephrine (ml/hr)', 'Phenylephrine (mcg/min)', 'Phenylephrine (mcg/kg/min)',
                 'Phenylephrine ()', 'Volume (ml) Phenylephrine ()', 'Phenylephrine (mg/hr)', 
                 'Phenylephrine  STD 20 mg Sodium Chloride 0.9% 250 ml (mcg/min)', 'Phenylephrine',
                 'Phenylephrine  MAX 100 mg Sodium Chloride 0.9% 250 ml (mcg/min)', 'Phenylephrine (mcg/hr)',
                 'Phenylephrine (mcg/min) (mcg/min)', 'Phenylephrine (mcg/kg/min) (mcg/kg/min)',
                 'Phenylephrine  STD 20 mg Sodium Chloride 0.9% 500 ml (mcg/min)', 'Phenylephrine (mg/kg/min)',
                 'Volume (ml) Phenylephrine'
                 'Neo-Synephrine (mcg/min)', 'neosynephrine (mcg/min)', 'Neosynephrine (mcg/min)',
                 'NeoSynephrine (mcg/min)', 'neosynsprine', 'neosynsprine (mcg/kg/hr)',
                 'NEO-SYNEPHRINE (mcg/min)', 'Neosynephrine (ml/hr)', 'neo-synephrine (mcg/min)',
                 'Neo Synephrine (mcg/min)']

Epinephrine = ['EPI (mcg/min)' , 'Epinepherine (mcg/min)' , 'Epinephrine' , 'Epinephrine ()',
               'EPINEPHrine(Adrenalin)MAX 30 mg Sodium Chloride 0.9% 250 ml (mcg/min)',
               'EPINEPHrine(Adrenalin)STD 4 mg Sodium Chloride 0.9% 250 ml (mcg/min)',
               'EPINEPHrine(Adrenalin)STD 4 mg Sodium Chloride 0.9% 500 ml (mcg/min)',
               'EPINEPHrine(Adrenalin)STD 7 mg Sodium Chloride 0.9% 250 ml (mcg/min)',
               'Epinephrine (mcg/hr)', 'Epinephrine (mcg/kg/min)', 'Epinephrine (mcg/min)', 'Epinephrine (mg/hr)',
               'Epinephrine (mg/kg/min)', 'Epinephrine (ml/hr)']

Dopamine = ['Dopamine (ml/hr)', 'Dopamine (mcg/kg/min)', 'Dopamine ()', 'Dopamine',
            'DOPamine STD 400 mg Dextrose 5% 250 ml  Premix (mcg/kg/min)',
            'DOPamine MAX 800 mg Dextrose 5% 250 ml  Premix (mcg/kg/min)', 'dopamine (mcg/kg/min)',
            'Dopamine (Unknown)', 'Dopamine (mcg/hr)', 'Dopamine (mcg/kg/hr)', 'Dopamine (mcg/min)',
            'DOPamine STD 15 mg Dextrose 5% 250 ml  Premix (mcg/kg/min)', 'Dopamine (mg/hr)',
            'DOPamine STD 400 mg Dextrose 5% 500 ml  Premix (mcg/kg/min)', 'Dopamine (nanograms/kg/min)']

Nicardipine = ['Nicardipine (ml/hr)', 'Nicardipine (mg/hr)', 'Nicardipine ()',
               'NiCARDipine (premix) 20 mg Sodium Chloride 0.9% 200 ml (mg/hr)',
               'NiCARDipine (std) 50 mg Sodium Chloride 0.9% 500 ml (mg/hr)',
               'NiCARDipine (max) 100 mg Sodium Chloride 0.9% 250 ml (mg/hr)',
               'Nicardipine (mcg/hr)', 'Nicardipine (mcg/kg/min)', 'nicardipine (mg/hr)', 
               'NiCARDipine (std) 50 mg  500 ml  Sodium Chloride 0.45% (mg/hr)', 
               'NiCARDipine (premix) 50 mg Sodium Chloride 0.9% 500 ml (mg/hr)', 
               'NiCARDipine (std) 20 mg Sodium Chloride 0.9% 200 ml (mg/hr)', 'Nicardipine (Unknown)']

Milrinone = ['Milrinone (ml/hr)', 'Milrinone (mcg/kg/min)', 'Milrinone ()', 
             'Milrinone (Primacor) 40 mg Dextrose 5% 200 ml (mcg/kg/min)', 'Milrinone (mcg/kg/hr)',
             'primacore (mcg/kg/min)', 'Milrinone', 'Milronone (mcg/kg/min)']

Pantoprazole = ['Pantoprazole (ml/hr)', 'Pantoprazole (mg/hr)', 'Pantoprazole ()', 
                'Pantoprazole(Protonix) (mg/hr)', 'Pantoprazole (mg/hr) (mg/hr)', 'Pantoprazole (mg/kg/hr)',
                'Pantoprazole (Protonix) 80 mg Sodium Chloride 0.9% 250 ml (mg/hr)',
                'Pantoprazole (Unknown)', 'Pantoprazole (mcg/kg/hr)']

Diltiazem = ['Diltiazem (ml/hr)', 'Diltiazem (mg/hr)', 'Diltiazem ()', 
             'Diltiazem (Cardizem) 125 mg Sodium Chloride 0.9% 125 ml (mg/hr)', 
             'Diltiazem(Cardizem) mg/hr (mg/hr)', 'Diltiazem (mg/kg/hr)', 
             'Diltiazem Inj 125 mg in Dextrose 5% 100 ml ()']

Dobutamine = ['Dobutamine (ml/hr)', 'Dobutamine (mcg/kg/min)', 'Dobutamine ()',
              'DOBUTamine STD 500 mg Dextrose 5% 250 ml  Premix (mcg/kg/min)',
              'DOBUTamine MAX 1000 mg Dextrose 5% 250 ml  Premix (mcg/kg/min)', 'Dobutamine (mcg/min)',
              'Dobutamine (mcg/kg/hr)', 'Dobutamine (units/min)']

Nitroglycerin = ['Nitroglycerin (ml/hr)', 'Nitroglycerin (mcg/min)', 'NitroGLYCERIN IVF Infused (ml/hr)',
                 'Nitroglycerin ()', 'Nitroglycerin (mcg/kg/min)', 'Nitroglycerin (Unknown)',
                 'Nitroglycerin(25mg/250ml) 25 mg Dextrose 5% 250 ml  Premix (glass bottle) (mcg/min)', 
                 'Nitroglycerin(50mg/500ml) 50 mg Dextrose 5% 500 ml  Premix (glass bottle) (mcg/min)',
                 'Nitroglycerin (mcg/kg/hr)', 'Nitroglycerin (mg/min)', 'Nitroglycerine (mcg/kg/min) (mcg/kg/min)']

In [79]:
infusionDrug = infusionDrug[infusionDrug.drugname.isin(Fentanyl)       | infusionDrug.drugname.isin(Propofol)     | 
                            infusionDrug.drugname.isin(Norepinephrine) | infusionDrug.drugname.isin(Insulin)      | 
                            infusionDrug.drugname.isin(Midazolam)      | infusionDrug.drugname.isin(Heparin)      | 
                            infusionDrug.drugname.isin(Dexmedetomidine)| infusionDrug.drugname.isin(Amiodarone)   | 
                            infusionDrug.drugname.isin(Vasopressin)    | infusionDrug.drugname.isin(Phenylephrine)|
                            infusionDrug.drugname.isin(Dopamine)       | infusionDrug.drugname.isin(Nicardipine)  |
                            infusionDrug.drugname.isin(Milrinone)      | infusionDrug.drugname.isin(Pantoprazole) |
                            infusionDrug.drugname.isin(Diltiazem)      | infusionDrug.drugname.isin(Dobutamine)   |
                            infusionDrug.drugname.isin(Nitroglycerin)  | infusionDrug.drugname.isin(Epinephrine)]

In [80]:
infusionDrug['itemvalue'] = 1
infusionDrug.rename(index=str, columns={"infusionoffset" : "itemoffset", "drugname" : "itemname"}, inplace=True)

In [81]:
infusionDrug.loc[infusionDrug.itemname.isin(Fentanyl)  ,       'itemname'] = 'Fentanyl_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Propofol)  ,       'itemname'] = 'Propofol_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Norepinephrine)  , 'itemname'] = 'Norepinephrine_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Insulin)  ,        'itemname'] = 'Insulin_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Midazolam)  ,      'itemname'] = 'Midazolam_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Heparin)  ,        'itemname'] = 'Heparin_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Dexmedetomidine),  'itemname'] = 'Dexmedetomidine_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Amiodarone)  ,     'itemname'] = 'Amiodarone_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Vasopressin)  ,    'itemname'] = 'Vasopressin_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Phenylephrine)  ,  'itemname'] = 'Phenylephrine_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Dopamine)  ,       'itemname'] = 'Dopamine_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Nicardipine)  ,    'itemname'] = 'Nicardipine_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Milrinone)  ,      'itemname'] = 'Milrinone_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Pantoprazole)  ,   'itemname'] = 'Pantoprazole_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Diltiazem)  ,      'itemname'] = 'Diltiazem_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Dobutamine)  ,     'itemname'] = 'Dobutamine_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Nitroglycerin)  ,  'itemname'] = 'Nitroglycerin_PRC'
infusionDrug.loc[infusionDrug.itemname.isin(Epinephrine)  ,    'itemname'] = 'Epinephrine_PRC'

In [82]:
infusionDrug = infusionDrug[infusionDrug.patientunitstayid.isin(cohort_icustay_id)]
infusionDrug = infusionDrug.reset_index(drop=True)

In [83]:
infusionDrug.head(3)

In [84]:
print(infusionDrug.shape)
print(infusionDrug.patientunitstayid.nunique())

(144228, 4)
4528


### medication

In [85]:
medication = pd.read_csv(eicu + "medication.csv")

<ipython-input>:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  medication = pd.read_csv(eicu + "medication.csv")


In [86]:
medication = medication[medication.drugorderoffset >= 0]
medication = medication[medication.drugordercancelled == 'No']
medication = medication[medication.dosage.notnull()]
medication = medication[['patientunitstayid', 'drugorderoffset', 'drugname', 'drughiclseqno']]

In [87]:
medication['itemname'] = 'Others'
medication['drugname'] = medication['drugname'].astype(str)
medication['drugname'] = medication['drugname'].apply(str.lower)

In [88]:
substring_antibiotics = ['adoxa', 'ala-tet', 'alodox', 'amikacin', 'amikin', 'amoxicill', 'amphotericin', 
                         'anidulafungin', 'ancef', 'clavulanate', 'ampicillin', 'augmentin', 'avelox', 'avidoxy', 
                         'azactam', 'azithromycin', 'aztreonam', 'axetil', 'bactocill', 'bactrim', 'bactroban', 
                         'bethkis', 'biaxin', 'bicillin l-a', 'cayston', 'cefazolin', 'cedax', 'cefoxitin', 
                         'ceftazidime', 'cefaclor', 'cefadroxil', 'cefdinir', 'cefditoren', 'cefepime', 'cefotan',
                         'cefotetan', 'cefotaxime', 'ceftaroline', 'cefpodoxime', 'cefpirome', 'cefprozil', 
                         'ceftibuten', 'ceftin', 'ceftriaxone', 'cefuroxime', 'cephalexin', 'cephalothin', 
                         'cephapririn', 'chloramphenicol', 'cipro', 'ciprofloxacin', 'claforan', 'clarithromycin',
                         'cleocin', 'clindamycin', 'cubicin', 'dicloxacillin', 'dirithromycin', 'doryx', 'doxycy',
                         'duricef', 'dynacin', 'ery-tab', 'eryped', 'eryc', 'erythrocin', 'erythromycin', 
                         'factive', 'flagyl', 'fortaz', 'furadantin', 'garamycin', 'gentamicin', 'kanamycin', 
                         'keflex', 'kefzol', 'ketek', 'levaquin', 'levofloxacin', 'lincocin', 'linezolid', 
                         'macrobid', 'macrodantin', 'maxipime', 'mefoxin', 'metronidazole', 'meropenem', 
                         'methicillin', 'minocin', 'minocycline', 'monodox', 'monurol', 'morgidox', 'moxatag', 
                         'moxifloxacin', 'mupirocin', 'myrac', 'nafcillin', 'neomycin', 'nicazel doxy 30',
                         'nitrofurantoin', 'norfloxacin', 'noroxin', 'ocudox', 'ofloxacin', 'omnicef', 'oracea', 
                         'oraxyl', 'oxacillin', 'pc pen vk', 'pce dispertab', 'panixine', 'pediazole', 'penicillin',
                         'periostat', 'pfizerpen', 'piperacillin', 'tazobactam', 'primsol', 'proquin', 'raniclor',
                         'rifadin', 'rifampin', 'rocephin', 'smz-tmp', 'septra', 'septra ds', 'septra', 'solodyn',
                         'spectracef', 'streptomycin', 'sulfadiazine', 'sulfamethoxazole', 'trimethoprim', 
                         'sulfatrim', 'sulfisoxazole', 'suprax', 'synercid', 'tazicef', 'tetracycline', 'timentin',
                         'tobramycin', 'trimethoprim', 'unasyn', 'vancocin', 'vancomycin', 'vantin', 'vibativ', 
                         'vibra-tabs', 'vibramycin', 'zinacef', 'zithromax', 'zosyn', 'zyvox']

for string in substring_antibiotics:
    medication.loc[medication['drugname'].str.contains(string),  'itemname'] = 'Antibiotic_PRC'

In [89]:
milrinone_HICL = [9744]
warfarin_HICL = [2812, 24859]
dobutamine_HICL = [8777, 40]
dopamine_HICL = [2060, 2059]
vasopressin_HICL = [38884, 38883, 2839]
norepinephrine_HICL = [37410, 36346, 2051]
phenylephrine_HICL = [37028, 35517, 35587, 2087]
epinephrine_HICL = [37407, 39089, 36437, 34361, 2050]
heparin_HICL = [39654, 9545, 2807, 33442, 8643, 33314, 2808, 2810]

medication.loc[medication['drughiclseqno'].isin(norepinephrine_HICL),'itemname'] = 'Norepinephrine_PRC'
medication.loc[medication['drughiclseqno'].isin(epinephrine_HICL),   'itemname'] = 'Epinephrine_PRC'
medication.loc[medication['drughiclseqno'].isin(dobutamine_HICL),    'itemname'] = 'Dobutamine_PRC'
medication.loc[medication['drughiclseqno'].isin(dopamine_HICL),      'itemname'] = 'Dopamine_PRC'
medication.loc[medication['drughiclseqno'].isin(phenylephrine_HICL), 'itemname'] = 'Phenylephrine_PRC'
medication.loc[medication['drughiclseqno'].isin(vasopressin_HICL),   'itemname'] = 'Vasopressin_PRC'
medication.loc[medication['drughiclseqno'].isin(milrinone_HICL),     'itemname'] = 'Milrinone_PRC'
medication.loc[medication['drughiclseqno'].isin(heparin_HICL),       'itemname'] = 'Heparin_PRC'
medication.loc[medication['drughiclseqno'].isin(warfarin_HICL),      'itemname'] = 'Warfarin_PRC'

In [90]:
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('heparin')),        'itemname'] = 'Heparin_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('warfarin')),       'itemname'] = 'Warfarin_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('dobutamine')),     'itemname'] = 'Dobutamine_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('dobutrex')),       'itemname'] = 'Dobutamine_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('norepinephrine')), 'itemname'] = 'Norepinephrine_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('levophed')),       'itemname'] = 'Norepinephrine_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('epinephrine')),    'itemname'] = 'Epinephrine_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('phenylephrine')),  'itemname'] = 'Phenylephrine_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('vasopressin')),    'itemname'] = 'Vasopressin_PRC'
medication.loc[(medication.drughiclseqno.isnull()) & (medication['drugname'].str.contains('milrinone')),      'itemname'] = 'Milrinone_PRC'

In [91]:
medication = medication[medication.itemname != 'Others']
medication.drop(columns=['drugname', 'drughiclseqno'], inplace=True)
medication['itemvalue'] = 1
medication.rename(index=str, columns={"drugorderoffset" : "itemoffset"}, inplace=True)
medication = medication[medication.patientunitstayid.isin(cohort_icustay_id)]
medication = medication.reset_index(drop=True)

In [92]:
medication.head(3)

In [93]:
print(medication.shape)
print(medication.patientunitstayid.nunique())

(22815, 4)
7204


### treatment 

In [94]:
treatment = pd.read_csv(eicu + "treatment.csv")

In [95]:
vasopressors_treatment = [ 
  'toxicology|drug overdose|vasopressors|vasopressin' #                                                                   |    23
, 'toxicology|drug overdose|vasopressors|phenylephrine (Neosynephrine)' #                                                 |    21
, 'toxicology|drug overdose|vasopressors|norepinephrine > 0.1 micrograms/kg/min' #                                        |    62
, 'toxicology|drug overdose|vasopressors|norepinephrine <= 0.1 micrograms/kg/min' #                                       |    29
, 'toxicology|drug overdose|vasopressors|epinephrine > 0.1 micrograms/kg/min' #                                           |     6
, 'toxicology|drug overdose|vasopressors|epinephrine <= 0.1 micrograms/kg/min' #                                          |     2
, 'toxicology|drug overdose|vasopressors|dopamine 5-15 micrograms/kg/min' #                                               |     7
, 'toxicology|drug overdose|vasopressors|dopamine >15 micrograms/kg/min' #                                                |     3
, 'toxicology|drug overdose|vasopressors' #                                                                               |    30
, 'surgery|cardiac therapies|vasopressors|vasopressin' #                                                                  |   356
, 'surgery|cardiac therapies|vasopressors|phenylephrine (Neosynephrine)' #                                                |  1000
, 'surgery|cardiac therapies|vasopressors|norepinephrine > 0.1 micrograms/kg/min' #                                       |   390
, 'surgery|cardiac therapies|vasopressors|norepinephrine <= 0.1 micrograms/kg/min' #                                      |   347
, 'surgery|cardiac therapies|vasopressors|epinephrine > 0.1 micrograms/kg/min' #                                          |   117
, 'surgery|cardiac therapies|vasopressors|epinephrine <= 0.1 micrograms/kg/min' #                                         |   178
, 'surgery|cardiac therapies|vasopressors|dopamine  5-15 micrograms/kg/min' #                                             |   274
, 'surgery|cardiac therapies|vasopressors|dopamine >15 micrograms/kg/min' #                                               |    23
, 'surgery|cardiac therapies|vasopressors' #                                                                              |   596
, 'renal|electrolyte correction|treatment of hypernatremia|vasopressin' #                                                 |     7
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|phenylephrine (Neosynephrine)' #           |   321
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|norepinephrine > 0.1 micrograms/kg/min' #  |   348
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|norepinephrine <= 0.1 micrograms/kg/min' # |   374
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|epinephrine > 0.1 micrograms/kg/min' #     |    21
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|epinephrine <= 0.1 micrograms/kg/min' #    |   199
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|dopamine 5-15 micrograms/kg/min' #         |   277
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors|dopamine > 15 micrograms/kg/min' #         |    20
, 'neurologic|therapy for controlling cerebral perfusion pressure|vasopressors' #                                         |   172
, 'gastrointestinal|medications|hormonal therapy (for varices)|vasopressin' #                                             |   964
, 'cardiovascular|shock|vasopressors|vasopressin' #                                                                       | 11082
, 'cardiovascular|shock|vasopressors|phenylephrine (Neosynephrine)' #                                                     | 13189
, 'cardiovascular|shock|vasopressors|norepinephrine > 0.1 micrograms/kg/min' #                                            | 24174
, 'cardiovascular|shock|vasopressors|norepinephrine <= 0.1 micrograms/kg/min' #                                           | 17467
, 'cardiovascular|shock|vasopressors|epinephrine > 0.1 micrograms/kg/min' #                                               |  2410
, 'cardiovascular|shock|vasopressors|epinephrine <= 0.1 micrograms/kg/min' #                                              |  2384
, 'cardiovascular|shock|vasopressors|dopamine  5-15 micrograms/kg/min' #                                                  |  4822
, 'cardiovascular|shock|vasopressors|dopamine >15 micrograms/kg/min' #                                                    |  1102
, 'cardiovascular|shock|vasopressors' #                                                                                   |  9335
, 'toxicology|drug overdose|agent specific therapy|beta blockers overdose|dopamine' #                                     |    66
, 'cardiovascular|ventricular dysfunction|inotropic agent|norepinephrine > 0.1 micrograms/kg/min' #                       |   537
, 'cardiovascular|ventricular dysfunction|inotropic agent|norepinephrine <= 0.1 micrograms/kg/min' #                      |   411
, 'cardiovascular|ventricular dysfunction|inotropic agent|epinephrine > 0.1 micrograms/kg/min' #                          |   274
, 'cardiovascular|ventricular dysfunction|inotropic agent|epinephrine <= 0.1 micrograms/kg/min' #                         |   456
, 'cardiovascular|shock|inotropic agent|norepinephrine > 0.1 micrograms/kg/min' #                                         |  1940
, 'cardiovascular|shock|inotropic agent|norepinephrine <= 0.1 micrograms/kg/min' #                                        |  1262
, 'cardiovascular|shock|inotropic agent|epinephrine > 0.1 micrograms/kg/min' #                                            |   477
, 'cardiovascular|shock|inotropic agent|epinephrine <= 0.1 micrograms/kg/min' #                                           |   505
, 'cardiovascular|shock|inotropic agent|dopamine <= 5 micrograms/kg/min' #                                                |  1103
, 'cardiovascular|shock|inotropic agent|dopamine  5-15 micrograms/kg/min' #                                               |  1156
, 'cardiovascular|shock|inotropic agent|dopamine >15 micrograms/kg/min' #                                                 |   144
, 'surgery|cardiac therapies|inotropic agent|dopamine <= 5 micrograms/kg/min' #                                           |   171
, 'surgery|cardiac therapies|inotropic agent|dopamine  5-15 micrograms/kg/min' #                                          |    93
, 'surgery|cardiac therapies|inotropic agent|dopamine >15 micrograms/kg/min' #                                            |     3
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|norepinephrine > 0.1 micrograms/kg/min' #              |   688
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|norepinephrine <= 0.1 micrograms/kg/min' #             |   670
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|epinephrine > 0.1 micrograms/kg/min' #                 |   381
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|epinephrine <= 0.1 micrograms/kg/min' #                |   357
, 'cardiovascular|ventricular dysfunction|inotropic agent|dopamine <= 5 micrograms/kg/min' #                              |   886
, 'cardiovascular|ventricular dysfunction|inotropic agent|dopamine  5-15 micrograms/kg/min' #                             |   649
, 'cardiovascular|ventricular dysfunction|inotropic agent|dopamine >15 micrograms/kg/min' #                               |    86
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|dopamine <= 5 micrograms/kg/min' #                     |   346
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|dopamine  5-15 micrograms/kg/min' #                    |   520
, 'cardiovascular|myocardial ischemia / infarction|inotropic agent|dopamine >15 micrograms/kg/min'] #                     |    54

In [96]:
treatment = treatment[treatment.treatmentoffset >= 0]
treatment = treatment[treatment.treatmentstring.isin(vasopressors_treatment)]
treatment.drop(columns=['treatmentid', 'activeupondischarge', 'treatmentstring'], inplace=True)
treatment['itemname'] = 'Vasopressors'
treatment['itemvalue'] = 1
treatment.rename(index=str, columns={"treatmentoffset" : "itemoffset"}, inplace=True)
treatment = treatment[treatment.patientunitstayid.isin(cohort_icustay_id)]
treatment = treatment.reset_index(drop=True)

In [97]:
treatment.head(3)

In [98]:
print(treatment.shape)
print(treatment.patientunitstayid.nunique())

(9227, 4)
2307


### Save csv files

In [99]:
patients.to_csv(output           + 'patients.csv' ,   index=False)
apache_df.to_csv(output          + 'apache.csv' ,     index=False)
pastHistory.to_csv(output        + 'pastHistory.csv', index=False)
lab.to_csv(output                + 'lab.csv'  ,       index=False)
respiratorySetting.to_csv(output + 'respiratorySetting.csv'  , index=False)
respiratoryData.to_csv(output    + 'respiratoryData.csv'  , index=False)
vitalAperiodic.to_csv(output     + 'vitalAperiodic.csv'  ,  index=False)
vitalPeriodic.to_csv(output      + 'vitalPeriodic.csv'  ,   index=False)
nurseCharting.to_csv(output      + 'nurseCharting.csv'  ,   index=False)
intakeOutput.to_csv(output       + 'intakeOutput.csv'  ,    index=False)
infusionDrug.to_csv(output       + 'infusionDrug.csv'  ,    index=False)
medication.to_csv(output         + 'medication.csv'  ,      index=False)
treatment.to_csv(output          + 'treatment.csv'  ,       index=False)

In [13]:
with open(output + 'ethnicity_dictionary.pkl', 'wb') as f:
    pickle.dump(ethnicity_dictionary, f)